# Phase 7 — SQL Analysis

This notebook builds and investigates a portable SQLite analytics layer over the five validated Phase 6 summary datasets. Raw, cleaned, and feature CSVs are treated as immutable inputs.

## 1. SQL analysis objectives

The analysis answers practical team, batting, bowling, and toss questions; demonstrates aggregation, joins, CTEs, conditional logic, subqueries, and window functions; and reconciles every core total to Phase 6. Sample thresholds are used for rate rankings so short careers do not produce misleading leaders.

In [1]:
from pathlib import Path
import hashlib
import sys

import pandas as pd
from IPython.display import Markdown, display

candidate = Path.cwd().resolve()
PROJECT_ROOT = candidate if (candidate / 'src').exists() else candidate.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import get_data_paths
from src.sql_analysis import (
    ANALYSIS_FILES, TABLE_FILES, connect_database, create_database,
    get_sql_paths, inspect_database, inspect_source_csvs,
    load_analysis_queries, parse_named_queries, query_dataframe,
)

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
data_paths = get_data_paths(PROJECT_ROOT)
sql_paths = get_sql_paths(PROJECT_ROOT)
protected_paths = [
    data_paths['matches_raw'], data_paths['deliveries_raw'],
    data_paths['matches_clean'], data_paths['deliveries_clean'],
    *[sql_paths['processed'] / filename for filename in TABLE_FILES.values()],
]
sha256 = lambda path: hashlib.sha256(path.read_bytes()).hexdigest()
source_hashes_before = {path.name: sha256(path) for path in protected_paths}
print('Portable project-relative configuration loaded.')

Portable project-relative configuration loaded.


## 2. Database creation and loading

The builder inspects all five CSVs, creates declared SQLite tables in a temporary database, loads the unchanged records, creates views, validates integrity, and atomically publishes the final database. A prior database is inspected before any rebuild.

In [2]:
display(inspect_source_csvs(PROJECT_ROOT))
database_path, database_validation = create_database(PROJECT_ROOT)
print('Database created:', database_path.relative_to(PROJECT_ROOT))
print(f"Build validation: {database_validation['status'].eq('PASS').sum()}/{len(database_validation)} PASS")

,table_name,rows,columns,null_cells,column_names
0,match_summary,1095,33,1146,"match_id, season_label, match_date, city, venu..."
1,innings_summary,2217,17,0,"match_id, season, inning, batting_team, bowlin..."
2,team_season_summary,146,19,0,"season, team, matches_played, decided_matches,..."
3,batting_summary,2617,14,381,"season, batter, matches, runs, balls_faced, fo..."
4,bowling_summary,1948,12,0,"season, bowler, matches, legal_balls, overs_bo..."


Database created: data\processed\ipl_analysis.db
Build validation: 49/49 PASS


## 3. Schema inspection

Natural keys and the match-to-innings relationship are declared in SQLite rather than inferred at query time.

In [3]:
display(inspect_database(database_path))
connection = connect_database(PROJECT_ROOT)
for table in TABLE_FILES:
    display(Markdown(f'**Schema: `{table}`**'))
    display(query_dataframe(connection, f'PRAGMA table_info("{table}")'))

,type,name
0,index,idx_batting_runs
1,index,idx_bowling_wickets
2,index,idx_innings_batting_team
3,index,idx_innings_bowling_team
4,index,idx_match_summary_season
5,index,idx_match_summary_teams
6,table,batting_summary
7,table,bowling_summary
8,table,innings_summary
9,table,match_summary


**Schema: `match_summary`**

,cid,name,type,notnull,dflt_value,pk
0,0,match_id,INTEGER,0,None,1
1,1,season_label,TEXT,1,None,0
2,2,match_date,TEXT,1,None,0
3,3,city,TEXT,1,None,0
4,4,venue,TEXT,1,None,0
5,5,team1,TEXT,1,None,0
6,6,team2,TEXT,1,None,0
7,7,toss_winner,TEXT,1,None,0
8,8,toss_decision,TEXT,1,None,0
9,9,winning_team,TEXT,0,None,0


**Schema: `innings_summary`**

,cid,name,type,notnull,dflt_value,pk
0,0,match_id,INTEGER,1,None,1
1,1,season,TEXT,1,None,0
2,2,inning,INTEGER,1,None,2
3,3,batting_team,TEXT,1,None,0
4,4,bowling_team,TEXT,1,None,0
5,5,recorded_deliveries,INTEGER,1,None,0
6,6,legal_balls,INTEGER,1,None,0
7,7,illegal_deliveries,INTEGER,1,None,0
8,8,total_runs,INTEGER,1,None,0
9,9,batsman_runs,INTEGER,1,None,0


...

,cid,name,type,notnull,dflt_value,pk
0,0,season,TEXT,1,None,1
1,1,team,TEXT,1,None,2
2,2,matches_played,INTEGER,1,None,0
3,3,decided_matches,INTEGER,1,None,0
4,4,wins,INTEGER,1,None,0
5,5,losses,INTEGER,1,None,0
6,6,ties,INTEGER,1,None,0
7,7,no_results,INTEGER,1,None,0
8,8,win_percentage,REAL,1,None,0
9,9,total_runs_scored,INTEGER,1,None,0


**Schema: `batting_summary`**

,cid,name,type,notnull,dflt_value,pk
0,0,season,TEXT,1,None,1
1,1,batter,TEXT,1,None,2
2,2,matches,INTEGER,1,None,0
3,3,runs,INTEGER,1,None,0
4,4,balls_faced,INTEGER,1,None,0
5,5,fours,INTEGER,1,None,0
6,6,sixes,INTEGER,1,None,0
7,7,boundary_runs,INTEGER,1,None,0
8,8,dot_balls,INTEGER,1,None,0
9,9,dismissals,INTEGER,1,None,0


**Schema: `bowling_summary`**

,cid,name,type,notnull,dflt_value,pk
0,0,season,TEXT,1,None,1
1,1,bowler,TEXT,1,None,2
2,2,matches,INTEGER,1,None,0
3,3,legal_balls,INTEGER,1,None,0
4,4,overs_bowled,REAL,1,None,0
5,5,runs_conceded,INTEGER,1,None,0
6,6,wickets,INTEGER,1,None,0
7,7,economy_rate,REAL,1,None,0
8,8,dot_balls,INTEGER,1,None,0
9,9,dot_ball_percentage,REAL,1,None,0


## 4. Table row counts

The database grain must exactly match each Phase 6 CSV.

In [4]:
reconciliation_queries = parse_named_queries(sql_paths['reconciliation'])
row_counts = query_dataframe(connection, reconciliation_queries['table_row_counts'])
display(row_counts)

,table_name,row_count
0,match_summary,1095
1,innings_summary,2217
2,team_season_summary,146
3,batting_summary,2617
4,bowling_summary,1948


## 5. Data-quality checks

SQLite integrity, foreign keys, row counts, cross-table totals, all named queries, and the query inventory are checked centrally.

In [5]:
display(database_validation)
assert database_validation['status'].eq('PASS').all()
queries = load_analysis_queries(PROJECT_ROOT)
query_count = sum(len(items) for items in queries.values())
print(f'Named analytical queries: {query_count}')
assert query_count == 30

,check,status,actual
0,SQLite integrity check,PASS,ok
1,Foreign-key check,PASS,0
2,match_summary row count,PASS,1095
3,innings_summary row count,PASS,2217
4,team_season_summary row count,PASS,146
5,batting_summary row count,PASS,2617
6,bowling_summary row count,PASS,1948
7,matches reconciliation,PASS,1095
8,total_runs reconciliation,PASS,347756
9,batter_runs reconciliation,PASS,330064


Named analytical queries: 30


In [6]:
def show_query(filename, name, question, interpretation, rows=15):
    query = queries[filename][name]
    result = query_dataframe(connection, query)
    display(Markdown(f'### {question}'))
    display(Markdown(f'```sql\n{query}\n```'))
    display(result.head(rows))
    display(Markdown(f'**Interpretation:** {interpretation}'))
    return result

## 6. Team performance analysis

Career totals, qualified percentages, season detail, scoring pace, and a transparent multi-metric profile are considered together.

In [7]:
team_wins = show_query('01_team_performance.sql', 'teams_by_total_wins', 'Which teams have the most total wins?', 'Mumbai Indians lead with 142 wins; totals measure longevity as well as performance.')
team_win_rate = show_query('01_team_performance.sql', 'qualified_teams_by_win_percentage', 'Which established teams have the highest win percentage?', 'Among teams with at least 50 decided matches, Chennai Super Kings lead at 58.47%.')
team_matches = show_query('01_team_performance.sql', 'team_matches_played', 'How many matches did each team play?', 'Participation totals use both match-team positions and preserve ties and no-results.')
team_seasons = show_query('01_team_performance.sql', 'team_season_performance', 'How did each team perform season by season?', 'The complete result supports season filters; rankings should retain each season’s different match volume.', rows=20)
team_scoring = show_query('01_team_performance.sql', 'qualified_teams_by_scoring_rate', 'Which teams have the highest aggregate scoring rate?', 'With at least 3,000 legal balls, Gujarat Titans lead at 8.83 runs per six legal balls; their shorter history still matters.')
team_profiles = show_query('01_team_performance.sql', 'strongest_overall_team_profiles', 'Which teams have the strongest overall statistical profiles?', 'The transparent four-rank composite places Mumbai Indians first and Chennai Super Kings second; it is not an official rating.')

### Which teams have the most total wins?

```sql
SELECT team, matches_played, wins, losses, ties, no_results, win_percentage
FROM v_team_overall
ORDER BY wins DESC, win_percentage DESC, team
LIMIT 15
```

,team,matches_played,wins,losses,ties,no_results,win_percentage
0,Mumbai Indians,261,142,115,4,0,55.25
1,Chennai Super Kings,238,138,98,1,1,58.47
2,Kolkata Knight Riders,251,130,117,4,0,52.63
3,Royal Challengers Bengaluru,255,121,128,3,3,48.59
4,Delhi Capitals,252,112,134,4,2,45.53
5,Rajasthan Royals,221,110,106,3,2,50.93
6,Punjab Kings,246,109,133,4,0,45.04
7,Sunrisers Hyderabad,182,87,91,4,0,48.88
8,Deccan Chargers,75,29,46,0,0,38.67
9,Gujarat Titans,45,28,17,0,0,62.22


**Interpretation:** Mumbai Indians lead with 142 wins; totals measure longevity as well as performance.

### Which established teams have the highest win percentage?

...

**Interpretation:** With at least 3,000 legal balls, Gujarat Titans lead at 8.83 runs per six legal balls; their shorter history still matters.

### Which teams have the strongest overall statistical profiles?

```sql
WITH scoring AS (
    SELECT batting_team AS team, SUM(total_runs) AS runs, SUM(legal_balls) AS legal_balls
    FROM innings_summary
    GROUP BY batting_team
), eligible AS (
    SELECT
        t.team, t.seasons_played, t.matches_played, t.wins, t.decided_matches,
        t.win_percentage, t.total_wickets_taken,
        ROUND(6.0 * s.runs / NULLIF(s.legal_balls, 0), 2) AS scoring_rate
    FROM v_team_overall AS t
    JOIN scoring AS s ON s.team = t.team
    WHERE t.decided_matches >= 50
), ranked AS (
    SELECT *,
        RANK() OVER (ORDER BY wins DESC) AS wins_rank,
        RANK() OVER (ORDER BY win_percentage DESC) AS win_rate_rank,
        RANK() OVER (ORDER BY scoring_rate DESC) AS scoring_rank,
        RANK() OVER (ORDER BY total_wickets_taken DESC) AS wickets_rank
    FROM eligible
)
SELECT *,
    ROUND((wins_rank + win_rate_rank + scoring_rank + wickets_rank) / 4.0, 2)
        AS composite_rank_score
FROM ranked
ORDER BY composite_rank_score, win_rate_rank, wins_rank, team
```

,team,seasons_played,matches_played,wins,decided_matches,win_percentage,total_wickets_taken,scoring_rate,wins_rank,win_rate_rank,scoring_rank,wickets_rank,composite_rank_score
0,Mumbai Indians,17,261,142,257,55.25,1458,8.37,1,2,3,1,1.75
1,Chennai Super Kings,15,238,138,236,58.47,1361,8.39,2,1,2,3,2.00
2,Royal Challengers Bengaluru,17,255,121,249,48.59,1364,8.42,4,6,1,2,3.25
3,Kolkata Knight Riders,17,251,130,247,52.63,1333,8.31,3,3,5,5,4.00
4,Rajasthan Royals,15,221,110,216,50.93,1155,8.24,6,4,7,7,6.00
5,Delhi Capitals,17,252,112,246,45.53,1347,8.19,5,7,8,4,6.00
6,Punjab Kings,17,246,109,242,45.04,1298,8.36,7,8,4,6,6.25
7,Sunrisers Hyderabad,12,182,87,178,48.88,984,8.29,8,5,6,8,6.75
8,Deccan Chargers,5,75,29,75,38.67,408,7.86,9,9,9,9,9.00


**Interpretation:** The transparent four-rank composite places Mumbai Indians first and Chennai Super Kings second; it is not an official rating.

## 7. Batting analysis

Career measures are recomputed from summed numerators and denominators. Rate and average leaderboards use explicit minimum samples.

In [8]:
bat_runs = show_query('02_batting_analysis.sql', 'top_15_run_scorers', 'Who are the top 15 run scorers?', 'V Kohli leads with 8,014 runs.')
bat_sr = show_query('02_batting_analysis.sql', 'qualified_strike_rate_leaders', 'Who has the highest qualified strike rate?', 'AD Russell leads at 174.84 among batters with at least 1,000 balls faced.')
bat_fours = show_query('02_batting_analysis.sql', 'most_fours', 'Who hit the most fours?', 'This ranks boundary events, with career runs used only as a tie-breaker.')
bat_sixes = show_query('02_batting_analysis.sql', 'most_sixes', 'Who hit the most sixes?', 'This is an event-volume ranking and should be read alongside career opportunity.')
bat_avg = show_query('02_batting_analysis.sql', 'qualified_batting_average_leaders', 'Who has the highest qualified batting average?', 'KL Rahul leads at 44.66 with the minimum set at 50 supported dismissals.')
bat_season = show_query('02_batting_analysis.sql', 'top_run_scorers_by_season', 'Who were the top run scorers in each season?', 'RANK preserves tied run totals and returns the top three ranks per season.', rows=20)
bat_combo = show_query('02_batting_analysis.sql', 'high_volume_high_strike_rate_batters', 'Who combines high run volume with high strike rate?', 'Among batters with 2,000 runs and 1,000 balls, AB de Villiers has the best average of run and strike-rate ranks.')

### Who are the top 15 run scorers?

```sql
SELECT batter, runs, balls_faced, dismissals, strike_rate, batting_average
FROM v_batting_career
ORDER BY runs DESC, strike_rate DESC, batter
LIMIT 15
```

,batter,runs,balls_faced,dismissals,strike_rate,batting_average
0,V Kohli,8014,6069,207,132.05,38.71
1,S Dhawan,6769,5326,192,127.09,35.26
2,RG Sharma,6630,5057,223,131.11,29.73
3,DA Warner,6567,4702,164,139.66,40.04
4,SK Raina,5536,4046,171,136.83,32.37
5,MS Dhoni,5243,3812,134,137.54,39.13
6,AB de Villiers,5181,3411,130,151.89,39.85
7,CH Gayle,4997,3346,126,149.34,39.66
8,RV Uthappa,4954,3801,180,130.33,27.52
9,KD Karthik,4843,3580,184,135.28,26.32


**Interpretation:** V Kohli leads with 8,014 runs.

### Who has the highest qualified strike rate?

...

**Interpretation:** RANK preserves tied run totals and returns the top three ranks per season.

### Who combines high run volume with high strike rate?

```sql
WITH qualified AS (
    SELECT *
    FROM v_batting_career
    WHERE runs >= 2000 AND balls_faced >= 1000
), ranked AS (
    SELECT *,
        RANK() OVER (ORDER BY runs DESC) AS run_rank,
        RANK() OVER (ORDER BY strike_rate DESC) AS strike_rate_rank
    FROM qualified
)
SELECT
    batter, runs, balls_faced, strike_rate, batting_average,
    run_rank, strike_rate_rank,
    ROUND((run_rank + strike_rate_rank) / 2.0, 2) AS combined_rank_score
FROM ranked
ORDER BY combined_rank_score, runs DESC, batter
LIMIT 20
```

,batter,runs,balls_faced,strike_rate,batting_average,run_rank,strike_rate_rank,combined_rank_score
0,AB de Villiers,5181,3411,151.89,39.85,7,4,5.50
1,CH Gayle,4997,3346,149.34,39.66,8,5,6.50
2,DA Warner,6567,4702,139.66,40.04,4,12,8.00
3,MS Dhoni,5243,3812,137.54,39.13,6,18,12.00
4,SK Raina,5536,4046,136.83,32.37,5,20,12.50
5,JC Buttler,3583,2430,147.45,37.72,20,8,14.00
6,KA Pollard,3437,2329,147.57,28.40,21,7,14.00
7,RR Pant,3297,2215,148.85,35.45,22,6,14.00
8,SV Samson,4419,3180,138.96,30.69,14,15,14.50
9,SA Yadav,3594,2474,145.27,31.81,19,10,14.50


**Interpretation:** Among batters with 2,000 runs and 1,000 balls, AB de Villiers has the best average of run and strike-rate ranks.

## 8. Bowling analysis

Economy uses legal balls and bowler-chargeable runs. Wickets exclude run outs and other non-bowler dismissals.

In [9]:
bowl_wickets = show_query('03_bowling_analysis.sql', 'top_15_wicket_takers', 'Who are the top 15 wicket takers?', 'YS Chahal leads the documented bowler-credit convention with 205 wickets.')
bowl_econ = show_query('03_bowling_analysis.sql', 'qualified_economy_leaders', 'Who has the best qualified economy?', 'M Muralitharan leads at 6.70 among bowlers with at least 1,200 legal balls.')
bowl_season = show_query('03_bowling_analysis.sql', 'top_wicket_takers_by_season', 'Who took the most wickets by season?', 'RANK preserves ties among the top three wicket ranks in every season.', rows=20)
bowl_combo = show_query('03_bowling_analysis.sql', 'high_wicket_good_economy_bowlers', 'Who combines 100 wickets with good economy?', 'The 100-wicket threshold balances sustained wicket volume with career economy.')
bowl_balls = show_query('03_bowling_analysis.sql', 'most_legal_balls_bowled', 'Who bowled the most legal balls?', 'Legal-ball workload is shown directly, with six-ball-equivalent overs for readability.')
bowl_profiles = show_query('03_bowling_analysis.sql', 'strongest_overall_bowling_profiles', 'Who has the strongest overall statistical bowling profile?', 'The wickets/economy/dot-ball composite highlights balanced careers but is not an official player rating.')

### Who are the top 15 wicket takers?

```sql
SELECT bowler, wickets, legal_balls, economy_rate, dot_ball_percentage
FROM v_bowling_career
ORDER BY wickets DESC, economy_rate, bowler
LIMIT 15
```

,bowler,wickets,legal_balls,economy_rate,dot_ball_percentage
0,YS Chahal,205,3521,7.84,33.91
1,PP Chawla,192,3850,7.96,34.42
2,DJ Bravo,183,3120,8.38,31.99
3,B Kumar,181,3910,7.56,41.74
4,SP Narine,180,4081,6.74,38.45
5,R Ashwin,180,4524,7.12,34.31
6,A Mishra,174,3371,7.38,35.15
7,SL Malinga,170,2828,7.14,40.45
8,JJ Bumrah,168,3075,7.30,39.93
9,RA Jadeja,160,3829,7.62,31.76


**Interpretation:** YS Chahal leads the documented bowler-credit convention with 205 wickets.

### Who has the best qualified economy?

...

**Interpretation:** Legal-ball workload is shown directly, with six-ball-equivalent overs for readability.

### Who has the strongest overall statistical bowling profile?

```sql
WITH qualified AS (
    SELECT * FROM v_bowling_career WHERE legal_balls >= 1200
), ranked AS (
    SELECT *,
        RANK() OVER (ORDER BY wickets DESC) AS wicket_rank,
        RANK() OVER (ORDER BY economy_rate) AS economy_rank,
        RANK() OVER (ORDER BY dot_ball_percentage DESC) AS dot_rank
    FROM qualified
)
SELECT
    bowler, wickets, legal_balls, economy_rate, dot_ball_percentage,
    wicket_rank, economy_rank, dot_rank,
    ROUND((wicket_rank + economy_rank + dot_rank) / 3.0, 2) AS composite_rank_score
FROM ranked
ORDER BY composite_rank_score, wicket_rank, bowler
LIMIT 20
```

,bowler,wickets,legal_balls,economy_rate,dot_ball_percentage,wicket_rank,economy_rank,dot_rank,composite_rank_score
0,SL Malinga,170,2828,7.14,40.45,8,7,16,10.33
1,DW Steyn,97,2182,6.94,46.70,26,4,1,10.33
2,B Kumar,181,3910,7.56,41.74,4,17,11,10.67
3,SP Narine,180,4081,6.74,38.45,5,2,28,11.67
4,JJ Bumrah,168,3075,7.30,39.93,9,9,19,12.33
5,Harbhajan Singh,150,3416,7.08,36.97,11,5,31,15.67
6,Rashid Khan,149,2872,6.83,36.91,12,3,32,15.67
7,A Mishra,174,3371,7.38,35.15,7,12,39,19.33
8,R Ashwin,180,4524,7.12,34.31,5,6,48,19.67
9,A Nehra,106,1908,7.85,41.82,22,29,10,20.33


**Interpretation:** The wickets/economy/dot-ball composite highlights balanced careers but is not an official player rating.

## 9. Toss analysis

Only ordinary runs/wickets outcomes enter win comparisons. These are descriptive associations and do not establish causality.

In [10]:
toss_overall = show_query('04_toss_analysis.sql', 'overall_toss_winner_match_win_rate', 'How often did the toss winner also win?', 'Toss winners also won 548 of 1,076 eligible matches (50.93%); this does not imply causation.')
toss_win = show_query('04_toss_analysis.sql', 'qualified_team_toss_win_percentage', 'Which teams won the toss most often relative to matches played?', 'Only teams with at least 50 matches are ranked to reduce small-sample distortion.')
toss_convert = show_query('04_toss_analysis.sql', 'qualified_team_toss_to_match_win_percentage', 'Which teams most often converted eligible toss wins into match wins?', 'Chennai Super Kings lead at 62.50% across 120 eligible toss wins.')
toss_season = show_query('04_toss_analysis.sql', 'toss_relationship_by_season', 'How did the toss relationship vary by season?', 'Season percentages fluctuate and should be interpreted as descriptive context.', rows=20)
toss_difference = show_query('04_toss_analysis.sql', 'team_toss_win_rate_difference', 'Which teams show the largest toss-win versus toss-loss difference?', 'The percentage-point difference requires 20 observations in each group and remains non-causal.')

### How often did the toss winner also win?

```sql
SELECT
    COUNT(*) AS eligible_matches,
    SUM(CASE WHEN toss_winner = winning_team THEN 1 ELSE 0 END) AS toss_winner_match_wins,
    ROUND(100.0 * SUM(CASE WHEN toss_winner = winning_team THEN 1 ELSE 0 END) / COUNT(*), 2)
        AS toss_winner_match_win_percentage
FROM match_summary
WHERE result IN ('runs', 'wickets')
```

,eligible_matches,toss_winner_match_wins,toss_winner_match_win_percentage
0,1076,548,50.93


**Interpretation:** Toss winners also won 548 of 1,076 eligible matches (50.93%); this does not imply causation.

### Which teams won the toss most often relative to matches played?

...

**Interpretation:** Season percentages fluctuate and should be interpreted as descriptive context.

### Which teams show the largest toss-win versus toss-loss difference?

```sql
WITH participation AS (
    SELECT match_id, team1 AS team, toss_winner, winning_team
    FROM match_summary WHERE result IN ('runs', 'wickets')
    UNION ALL
    SELECT match_id, team2 AS team, toss_winner, winning_team
    FROM match_summary WHERE result IN ('runs', 'wickets')
), team_split AS (
    SELECT
        team,
        SUM(CASE WHEN team = toss_winner THEN 1 ELSE 0 END) AS toss_won_matches,
        SUM(CASE WHEN team = toss_winner AND team = winning_team THEN 1 ELSE 0 END) AS wins_after_toss_win,
        SUM(CASE WHEN team <> toss_winner THEN 1 ELSE 0 END) AS toss_lost_matches,
        SUM(CASE WHEN team <> toss_winner AND team = winning_team THEN 1 ELSE 0 END) AS wins_after_toss_loss
    FROM participation
    GROUP BY team
)
SELECT
    team, toss_won_matches, wins_after_toss_win,
    ROUND(100.0 * wins_after_toss_win / NULLIF(toss_won_matches, 0), 2) AS win_rate_after_toss_win,
    toss_lost_matches, wins_after_toss_loss,
    ROUND(100.0 * wins_after_toss_loss / NULLIF(toss_lost_matches, 0), 2) AS win_rate_after_toss_loss,
    ROUND(
        100.0 * wins_after_toss_win / NULLIF(toss_won_matches, 0)
        - 100.0 * wins_after_toss_loss / NULLIF(toss_lost_matches, 0), 2
    ) AS descriptive_percentage_point_difference
FROM team_split
WHERE toss_won_matches >= 20 AND toss_lost_matches >= 20
ORDER BY descriptive_percentage_point_difference DESC, team
```

,team,toss_won_matches,wins_after_toss_win,win_rate_after_toss_win,toss_lost_matches,wins_after_toss_loss,win_rate_after_toss_loss,descriptive_percentage_point_difference
0,Deccan Chargers,43,19,44.19,32,10,31.25,12.94
1,Chennai Super Kings,120,75,62.50,116,63,54.31,8.19
2,Kolkata Knight Riders,121,68,56.20,126,62,49.21,6.99
3,Royal Challengers Bengaluru,118,60,50.85,131,61,46.56,4.28
4,Gujarat Titans,22,14,63.64,23,14,60.87,2.77
5,Delhi Capitals,127,59,46.46,119,53,44.54,1.92
6,Mumbai Indians,140,77,55.00,117,65,55.56,-0.56
7,Rajasthan Royals,117,59,50.43,99,51,51.52,-1.09
8,Punjab Kings,107,44,41.12,135,65,48.15,-7.03
9,Sunrisers Hyderabad,87,38,43.68,91,49,53.85,-10.17


**Interpretation:** The percentage-point difference requires 20 observations in each group and remains non-causal.

## 10. Advanced SQL and window functions

The queries below demonstrate season partitions, tied ranks, deterministic top rows, season-average comparisons, and lagged team results.

In [11]:
advanced_context = {
    'team_rank_within_season': ('How do teams rank within each season?', 'RANK uses wins then win percentage within each season.'),
    'batter_rank_within_season': ('How do batters rank by season runs?', 'The season partition prevents cross-season volume comparisons.'),
    'bowler_rank_within_season': ('How do bowlers rank by season wickets?', 'Tied wicket totals receive the same rank.'),
    'top_batter_each_season': ('Who is the deterministic top run scorer in each season?', 'ROW_NUMBER selects one row using strike rate and name as tie-breakers.'),
    'team_vs_season_average': ('Which teams were above their season average?', 'A window average supplies context without collapsing team rows.'),
    'team_wins_year_over_year': ('How did team win totals change from their prior recorded season?', 'LAG compares adjacent recorded seasons; schedule size and missed seasons remain limitations.'),
}
advanced_results = {}
for query_name, (question, interpretation) in advanced_context.items():
    advanced_results[query_name] = show_query('05_advanced_analysis.sql', query_name, question, interpretation, rows=18)

### How do teams rank within each season?

```sql
SELECT
    season, team, matches_played, wins, win_percentage,
    RANK() OVER (
        PARTITION BY season
        ORDER BY wins DESC, win_percentage DESC
    ) AS season_team_rank
FROM team_season_summary
ORDER BY season, season_team_rank, team
```

,season,team,matches_played,wins,win_percentage,season_team_rank
0,2007/08,Rajasthan Royals,16,13,81.25,1
1,2007/08,Punjab Kings,15,10,66.67,2
2,2007/08,Chennai Super Kings,16,9,56.25,3
3,2007/08,Delhi Capitals,14,7,50.00,4
4,2007/08,Mumbai Indians,14,7,50.00,4
5,2007/08,Kolkata Knight Riders,13,6,46.15,6
6,2007/08,Royal Challengers Bengaluru,14,4,28.57,7
7,2007/08,Deccan Chargers,14,2,14.29,8
8,2009,Delhi Capitals,15,10,66.67,1
9,2009,Deccan Chargers,16,9,56.25,2


**Interpretation:** RANK uses wins then win percentage within each season.

### How do batters rank by season runs?

...

**Interpretation:** A window average supplies context without collapsing team rows.

### How did team win totals change from their prior recorded season?

```sql
WITH sequenced AS (
    SELECT
        season, team, wins,
        LAG(wins) OVER (
            PARTITION BY team
            ORDER BY CAST(SUBSTR(season, 1, 4) AS INTEGER), season
        ) AS previous_season_wins
    FROM team_season_summary
)
SELECT
    season, team, wins, previous_season_wins,
    wins - previous_season_wins AS change_in_wins
FROM sequenced
ORDER BY team, CAST(SUBSTR(season, 1, 4) AS INTEGER), season
```

,season,team,wins,previous_season_wins,change_in_wins
0,2007/08,Chennai Super Kings,9,NaN,NaN
1,2009,Chennai Super Kings,8,9.00,-1.00
2,2009/10,Chennai Super Kings,9,8.00,1.00
3,2011,Chennai Super Kings,11,9.00,2.00
4,2012,Chennai Super Kings,10,11.00,-1.00
5,2013,Chennai Super Kings,12,10.00,2.00
6,2014,Chennai Super Kings,10,12.00,-2.00
7,2015,Chennai Super Kings,10,10.00,0.00
8,2018,Chennai Super Kings,11,10.00,1.00
9,2019,Chennai Super Kings,10,11.00,-1.00


**Interpretation:** LAG compares adjacent recorded seasons; schedule size and missed seasons remain limitations.

## 11. Reconciliation against Phase 6

These anchors test the full SQL load rather than isolated sample results.

In [12]:
totals = query_dataframe(connection, reconciliation_queries['reconciliation_totals'])
expected = {'matches': 1095, 'total_runs': 347756, 'batter_runs': 330064, 'extras': 17692, 'legal_balls': 251471, 'bowler_credit_wickets': 11815}
comparison = pd.DataFrame({
    'metric': list(expected),
    'sql_result': [int(totals.loc[0, key]) for key in expected],
    'phase_6_anchor': list(expected.values()),
})
comparison['difference'] = comparison['sql_result'] - comparison['phase_6_anchor']
comparison['status'] = comparison['difference'].eq(0).map({True: 'PASS', False: 'FAIL'})
display(comparison)
assert comparison['status'].eq('PASS').all()

,metric,sql_result,phase_6_anchor,difference,status
0,matches,1095,1095,0,PASS
1,total_runs,347756,347756,0,PASS
2,batter_runs,330064,330064,0,PASS
3,extras,17692,17692,0,PASS
4,legal_balls,251471,251471,0,PASS
5,bowler_credit_wickets,11815,11815,0,PASS


## 12. Key findings

- Mumbai Indians have the most wins (142); Chennai Super Kings lead qualified career win percentage (58.47%).
- V Kohli leads runs (8,014), AD Russell leads qualified strike rate (174.84), and KL Rahul leads qualified batting average (44.66).
- YS Chahal leads bowler-credit wickets (205); M Muralitharan leads qualified economy (6.70).
- Toss winners also won 50.93% of the 1,076 ordinary decided matches. This is descriptive, not causal.
- All conclusions are conditional on the documented Phase 4 identity mappings, Phase 6 cricket conventions, and Phase 7 sample thresholds.

## 13. Limitations

Player names are not stable IDs; player match counts are role appearances rather than squad appearances; super overs remain included; historical franchise identity follows the existing standardization; composite ranks depend on selected dimensions; and toss comparisons cannot establish causal effects.

## 14. Source integrity and conclusion

The final hash comparison verifies that database generation did not alter any raw, cleaned, or Phase 6 analytical CSV.

In [13]:
source_hashes_after = {path.name: sha256(path) for path in protected_paths}
integrity = pd.DataFrame({
    'source_file': list(source_hashes_before),
    'unchanged': [source_hashes_before[name] == source_hashes_after[name] for name in source_hashes_before],
})
display(integrity)
assert integrity['unchanged'].all()
connection.close()
print('Phase 7 SQL analysis completed successfully.')
print('Tableau work has not started.')

,source_file,unchanged
0,matches.csv,True
1,deliveries.csv,True
2,matches_clean.csv,True
3,deliveries_clean.csv,True
4,match_summary.csv,True
5,innings_summary.csv,True
6,team_season_summary.csv,True
7,batting_summary.csv,True
8,bowling_summary.csv,True


Phase 7 SQL analysis completed successfully.
Tableau work has not started.
